# 10 · Setup and correctness checks  (~20–40 min)

> Part of the October 2026 experiments (notebooks 10–19), run with the fastapriori 0.2.0 build and the `bench/exp_runners.py` harness. Defaults: 1 warm-up + 3 measured runs per cell, full repetitions even for slow cells, explicit thread counts, failures recorded. Never run two of these notebooks at the same time.

Run first, once per VM session. Feeds: paper §5 *Size bounds* (A1) and the LCM
invocation in the artifact.

1. builds the Rust extension and runs the variant correctness tests;
2. preflight of every method the other notebooks use;
3. **LCM template check**: reruns LCM on Chainstore (s=1e-4, k=2,3) and compares the
   per-level counts with the May 2026 LCM results (|F1|=12,194, |F2|=19,945, |F3|=3,139);
4. **A1 bound check**: Borgelt binaries with `-m k -n k` against `-m k` only.

Outputs: `results/runs/<campaign>/a1_bounds.csv`, `results/runs/<campaign>/lcm_check.csv`.

In [ ]:
# The library is normally installed by setup_vm.sh (pip wheel or source build of fastapriori 0.2.0).
# Set BUILD = True to rebuild it here from lib/fastapriori-0.2.0 instead.
import subprocess, sys
from pathlib import Path
assert sys.platform.startswith("linux"), (
    "10-19 time every run with /usr/bin/time and ulimit: run them on the VM (or WSL).")
BUILD = False
LIB = Path.cwd().resolve().parent / "lib" / "fastapriori-0.2.0"
if BUILD:
    b = subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", str(LIB)],
                       capture_output=True, encoding="utf-8", errors="replace")
    print(b.stdout[-600:], b.stderr[-1200:]); assert b.returncode == 0, "build failed"
import fastapriori, importlib.metadata as md
print("fastapriori", md.version("fastapriori"), "from", fastapriori.__file__)

In [ ]:
import os, sys, json, math, time
from pathlib import Path
BUNDLE = Path.cwd().resolve().parent     # notebooks run from <bundle>/notebooks
assert (BUNDLE / "bench" / "exp_runners.py").exists(), "start Jupyter in <bundle>/notebooks"
sys.path.insert(0, str(BUNDLE / "bench"))
# Results of these notebooks go to results/runs/<FA_CAMPAIGN>/ (set it before starting Jupyter
# to keep campaigns apart; every run_grid call also appends to manifest.jsonl there).
os.environ.setdefault("FA_CAMPAIGN", "new_campaign")
import numpy as np, pandas as pd
import exp_runners as pr
import exp_data as pdd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print("campaign", pr.CAMPAIGN, "->", pr.RESULTS)
print("build", pr.build_id(), "| extension", pr.ext_path())

In [ ]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--import-mode=importlib", "-p", "no:cacheprovider", str(pr.REPO_DIR / "tests")],
                   capture_output=True, encoding="utf-8", errors="replace")
print(r.stdout[-1500:]); assert r.returncode == 0, "tests failed: do not start the VM runs"

In [ ]:
ALL = ["hyb", "hyb_1t", "hyb_sfx", "hyb_sfx_1t", "hyb_seeded", "hyb_seeded_1t", "hyb_eclat",
       "hyb_prefilter", "classic", "classic_1t", "pyfim_apriori", "pyfim_eclat", "pyfim_fpgrowth",
       "borgelt_apriori", "borgelt_eclat", "borgelt_fpgrowth", "lcm",
       "hybcli_load", "hybcli_pairs_v0", "hybcli_pairs_v2", "hybcli_anchor", "hybcli_seeded"]
pr.preflight(ALL)
print(pr.br.collect_environment())

## Smoke grid (~2 min)
Every method once on Groceries at k=2,3. All rows must be `ok` and the size-k counts must
agree before any long notebook is started. Delete `results/runs/<campaign>/smoke.csv` to rerun.

In [ ]:
PAIRS_ONLY = [m for m in ALL if m.startswith("hybcli_pairs")]
kw = dict(supports=[1e-3], repeats=1, warmup=0, timeout_s=300, cascade=False)
K3_ONLY = ["hybcli_anchor"]
pr.run_grid("smoke", ["Groceries"], PAIRS_ONLY, ks=[2], **kw)
pr.run_grid("smoke", ["Groceries"], K3_ONLY, ks=[3], **kw)
smoke = pr.run_grid("smoke", ["Groceries"], [m for m in ALL if m not in PAIRS_ONLY + K3_ONLY + ["lcm"]],
                    ks=[2, 3], **kw)
print(smoke.groupby(["method", "status"]).size().unstack(fill_value=0))
assert (smoke.status == "ok").all(), smoke[smoke.status != "ok"][["method", "k", "status", "stderr_tail"]]
bad = pr.check_agreement(pr.summarize(smoke).assign(n_itemsets=lambda d: d.n_itemsets))
assert bad.empty, bad

## LCM invocation
The LCM driver of the May 2026 runs was not archived. Check the usage text below and set
`LCM_ARGS` (environment variable or `pr.LCM_ARGS`) so the check below reproduces the May 2026
counts. The October 2026 campaign used `F -u {k} {input} {min_count}` (recorded in `manifest.jsonl`).

In [ ]:
print(subprocess.run([pr.LCM_BIN], capture_output=True, text=True).stdout[:2500])
print("current template:", pr.LCM_BIN, pr.LCM_ARGS)

In [ ]:
rows = []
for k in (2, 3):
    r = pr.run_once("lcm", "Chainstore", k, 1e-4)
    rows.append(dict(k=k, status=r["status"], wall_s=r["wall_s"], counts=r["lcm_counts"], n_k=r["n_itemsets"]))
chk = pd.DataFrame(rows); chk.to_csv(pr.RESULTS / "lcm_check.csv", index=False); chk
# Expect counts to contain 12194 (size 1), 19945 (size 2) and, for k=3, 3139 (size 3).

## A1: do Borgelt's size bounds act inside the search?
If `-n k` only filtered output, both variants would take the same time. A large gap
(nobound slower) shows the bound prunes the search. Counts differ by design: without
`-n` the binary also reports sets larger than k.

In [ ]:
A1_DATASETS = ["BMS-WebView-2", "Retail (Belgian)", "Chainstore"]
A1_METHODS = ["borgelt_apriori", "borgelt_apriori_nobound", "borgelt_eclat", "borgelt_eclat_nobound",
              "borgelt_fpgrowth", "borgelt_fpgrowth_nobound"]
pr.RESULTS.mkdir(parents=True, exist_ok=True)
res = pr.run_grid("a1_bounds", A1_DATASETS, A1_METHODS, ks=[2, 3], supports=[1e-4],
                  timeout_s=900, cascade=False)
s = pr.summarize(res)
s.pivot_table(index=["dataset", "k"], columns="method", values="wall_s")